# Knowledge-Graph Embeddings: TransE and RotatE

The narrative companion to formalNLP's twenty-fourth topic. Every number here comes from
`knowledge_graph_embeddings.py`, whose harness asserts each claim the page makes. The testbed is the finance corpus's
knowledge base `WORLD` (114 entities, 10 relations, 254 triples), built so that its relations carry the patterns
embedding models are judged by: COMPETES_WITH is symmetric, PARENT_OF and SUBSIDIARY_OF are inverses, and
BASED_IN = HQ_IN ∘ LOCATED_IN. Every model trains against every corruption of every triple (the exact expectation of
uniform negative sampling), so a run is deterministic given its initialization.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/knowledge-graph-embeddings").resolve()))
import knowledge_graph_embeddings as kge

np.set_printoptions(precision=4, suppress=True)
print(len(kge.ENTITIES), "entities,", len(kge.RELATIONS), "relations,", len(kge.WORLD.triples), "triples")
print(Counter(kge.WORLD.types.values()))
print(Counter(r for _, r, _ in kge.WORLD.triples))

## 1. Link prediction (Def 1, Def 2)

A query $(h, r, ?)$ ranks every entity by the score $f(h, r, t)$. The filtered rank of the true answer drops the
query's other known answers and splits ties evenly (the realistic rank, the mean of the optimistic and pessimistic
ranks). MRR averages its reciprocal and Hits@$k$ the share of ranks $\le k$; both directions of a test triple are
queries. The harness checks the realistic rank against an enumeration over tie-breaking orders.

In [ ]:
kge.test_def2_realistic_rank_is_the_mean_over_tie_breaks()
s = np.array([3.0, 5.0, 3.0, 3.0, 1.0])
print("answer 0 among scores", s, "with entity 1 a known answer:", kge.realistic_rank(s, 0, [1]))
print(kge.metrics([1, 2, 4, 10, 11]))

## 2. Translation: TransE (Def 3, Prop 1, Prop 2)

TransE scores $-\lVert e_h + w_r - e_t \rVert$. Translations invert ($w' = -w$) and compose ($w_3 = w_1 + w_2$)
exactly, but a symmetric pair holds exactly only at $w = 0$, where the pair collapses. Without the unit-norm
constraint, scaling every vector shrinks Bordes et al.'s margin loss to zero on any embedding that separates its
facts, while no rank moves.

In [ ]:
kge.test_prop1_translations_invert_and_compose_exactly()
kg, E, R = kge.planted_transe()
P = kge.problem(kg)
for s in (0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    print(f"scale {s:5.2f}: margin loss {kge.margin_loss(s * E, s * R, P, 1.0):.6f}")
kge.test_prop2_scaling_alone_drives_the_margin_loss_to_zero()

## 3. Rotation: RotatE (Lemma 1, Def 4, Thm 1)

Multiplying by $e^{i\theta}$ rotates the plane by $\theta$. RotatE scores $-\lVert h \circ r - t \rVert$ with
$|r_i| = 1$, so every relation pattern becomes an equation in the phases: symmetric iff $\theta_i \in \{0, \pi\}$,
inverse iff $\theta' = -\theta$, composed iff the phases add. Trained on the whole KB, the learned phases solve those
equations almost exactly.

In [ ]:
kge.test_lemma1_multiplying_by_a_unit_complex_number_rotates()
kge.test_thm1_rotation_patterns_are_phase_equations()
res = kge.phase_residuals(kge.rotate_full().R, kge.FULL_KG.rid)
for k, v in res.items():
    print(f"{k:12s} median residual {np.median(v):.6f}   (uniform phases: {kge.UNIFORM_MEDIAN[k]:.6f})")

Hide facts and the picture changes. On split 0 one direction of every COMPETES_WITH pair is hidden, and a hidden
fact is a *negative* during training: the closed-world assumption says it is false. With phases at $\{0, \pi\}$ a
rotation scores $(a, r, b)$ exactly as $(b, r, a)$, so to push the hidden direction down the model moves its phases
off the symmetric solutions. It learns the false antisymmetry the incomplete KB teaches.

In [ ]:
res = kge.phase_residuals(kge.fit(kge.chosen("RotatE"), kge.pattern_split(0)[0]).R, kge.FULL_KG.rid)
for k, v in res.items():
    print(f"{k:12s} median residual {np.median(v):.6f}")
kge.test_thm1_trained_phases_obey_the_equations_and_incompleteness_breaks_them()

## 4. Relations as group actions (Def 5, Thm 2, Thm 3, Cor 1)

Both models act on entity space by isometries: TransE by the translation group $\mathbb{R}^k$, RotatE by the torus
$U(1)^k$. Isometry alone forces collapse: an N-to-1 relation's heads sit within $2\varepsilon$ of one another, and on
an odd cycle of a symmetric relation neighbors sit within $2L\varepsilon$. The finance KB's competitor graph has 8
triangles.

In [ ]:
cg = kge.competitor_graph()
print("competitor degrees", dict(cg["degrees"]), "triangles", len(cg["triangles"]))
kge.test_thm2_isometries_collapse_n_to_1_heads_symmetric_pairs_and_odd_cycles()

Both groups are abelian, so neither can tell $ab$ from $ba$. On the Cayley graph of $S_3$ (a transposition $a$, a
3-cycle $b$), every corruption $(xa, B, xba)$ sits within $5\varepsilon$ of its query, so no abelian model keeps its
corruptions farther than $5\varepsilon$. The dihedral group $D_3$, the symmetries of a triangle, acts by a reflection
and a rotation and fits the graph exactly.

In [ ]:
for m in kge.DISTANCE:
    for k in kge.S3_K:
        b = kge.s3_best(m, k)
        print(f"{m:7s} k={k}: eps {b['eps']:.3f}, nearest corruption {b['gamma_min']:.3f}, ratio {b['ratio']:.3f}")
d = kge.dihedral_fit()
print("D3: max fact error", d["max_fact_error"], "separation", round(d["separation"], 6), "det", d["det"])
kge.test_thm3_abelian_models_cannot_separate_s3s_corruptions()
kge.test_cor1_the_dihedral_group_fits_s3_exactly()

## 5. The bilinear foil (Def 6, Props 3–5)

DistMult scores $\langle h, r, t\rangle$, symmetric in $h$ and $t$ for every relation. ComplEx scores
$\mathrm{Re}\langle h, r, \bar t\rangle$, whose antisymmetric part is $-2 \sum_i \mathrm{Im}\, r_i\,
\mathrm{Im}(h_i \bar t_i)$. RotatE's squared distance is ComplEx's score with unit-modulus relations, plus the two
entity norms.

In [ ]:
kge.test_props3to5_bilinear_identities()
for m in kge.MODELS:
    print(f"{m:8s} direction accuracy {np.mean(kge.pattern_results(m)['direction']):.6f}")

## 6. Finance case study: patterns held out

Split 0 hides one direction of every COMPETES_WITH pair (symmetry), one fact of every parent–subsidiary pair
(inverse), half the BASED_IN facts (composition), and every subsidiary's sector (inheritance, a regularity of the
world rather than a rule of the KB). Hyperparameters were chosen on split 1, averaged over three seeds, on a grid of the margin or penalty and of the step count (250 to 1,500).

In [ ]:
for m in kge.MODELS:
    c = kge.chosen(m)
    print(f"{m:8s} gamma {c.gamma:g} lam {c.lam:g} steps {c.steps}")
print()
print(f"{'':10s}" + "".join(f"{p:>14s}" for p in kge.PATTERNS))
for m in kge.MODELS:
    res = kge.pattern_results(m)
    print(f"{m:10s}" + "".join(f"{np.mean([x['mrr'] for x in res[p]]):14.3f}" for p in kge.PATTERNS))
kge.test_tuning_picks_the_validation_argmax()
kge.test_pattern_results_the_page_reports()

## 7. Finance case study: completing the extracted KB

Topic 23's pipeline read the held-out companies' filings and extracted 45 facts above 0.5, all true. Beside the
training roster's 188 facts, they leave 28 WORLD facts about the held-out entities missing: 26 competitor facts and 2
SUBSIDIARY_OF facts whose PARENT_OF inverse was extracted. At a threshold of 0.1 the KB gains 19 facts, 17 of them
false.

In [ ]:
for tau in kge.THRESHOLDS:
    cp = kge.completion_problem(tau)
    print(f"tau {tau}: {len(cp['extracted'])} extracted ({cp['true']} true), missing", Counter(r for _, r, _ in cp["missing"]))
    for m in kge.MODELS:
        out = {r: (np.mean([x['mrr'] for x in v]), np.mean([x['h10'] for x in v]))
               for r, v in kge.completion_results(m, tau).items()}
        print(f"   {m:8s}", {r: f"MRR {a:.3f} H@10 {b:.3f}" for r, (a, b) in out.items()})
print("the overview example:", kge.lab_example(), round(kge.extracted()[kge.lab_example()], 6))
kge.test_completing_the_extracted_kb()

## The harness

Every claim above, plus the KB's sizes, the gradient checks, and the lab's data file.

In [ ]:
kge.test_kb_as_the_page_describes_it()
kge.test_gradients_match_central_differences()
kge.test_viz_mirror_matches_payload()
print("all checks passed")